In [22]:
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

load_dotenv()
# hf_token = os.getenv("HUGGINGFACE_API_KEY")
Gapikey=os.getenv("GOOGLE_API_KEY")


# 2. Set task to 'conversational' to satisfy the provider (featherless-ai)
# llm = HuggingFaceEndpoint(
#     repo_id="google/gemma-3-27b-it",
#     task="conversational", 
#     max_new_tokens=512,
#     huggingfacehub_api_token=hf_token
# )

model=ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", api_key=Gapikey,timeout=60)


In [2]:
# Test 

# from langchain_google_genai import ChatGoogleGenerativeAI

# model = ChatGoogleGenerativeAI(
#     model="gemini-1.5-flash",
#     temperature=0,
#     timeout=60
# )

response = model.invoke("Say hello")
print(response.content)


Hello! How can I help you today?


In [3]:
# Tool creation

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers and return product."""
    return a * b

In [4]:
print(multiply.invoke({"a": 6, "b": 10}))

60


In [36]:
# tool biding

In [5]:
llm_with_tool=model.bind_tools([multiply])

In [6]:
query= "can you multiply 6 and 10 for me?"

In [7]:
messages=[HumanMessage(content=query)]

In [8]:
print(messages)

[HumanMessage(content='can you multiply 6 and 10 for me?', additional_kwargs={}, response_metadata={})]


In [13]:
result=llm_with_tool.invoke("can you multiply 5 and 10?")

In [14]:
messages.append(result)

In [17]:
print(messages)

[HumanMessage(content='can you multiply 6 and 10 for me?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 10, "a": 5}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c48a6-e9b4-7263-96ff-4895d927f43e-0', tool_calls=[{'name': 'multiply', 'args': {'b': 10, 'a': 5}, 'id': 'f6784ebc-15a8-4b2e-bc52-eb7f42614a68', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 19, 'total_tokens': 80, 'input_token_details': {'cache_read': 0}}), AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 10, "a": 5}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c48b4-953a-7480-ad75-1c115f41ecd3-0', tool_ca

In [19]:
tool_result=multiply.invoke(result.tool_calls[0])

In [20]:
messages.append(tool_result)

In [ ]:
# https://colab.research.google.com/drive/1-xMYU9ExZqoySEX-XHAvEaE17PCWvc9H?usp=sharing

In [ ]:
# Creating a currency conversion tool

In [90]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated
from langchain_core.messages import ToolMessage
@tool
def get_conversion_rate(from_currency: str, to_currency: str) -> float:
    """Returns the conversion rate from one currency to another."""
    # For demonstration purposes, we'll return a fixed conversion rate.
    # Featching currency from API

    url=f"https://v6.exchangerate-api.com/v6/8fa2023b39be5389fc9a6a5f/pair/{from_currency}/{to_currency}"

    # 8fa2023b39be5389fc9a6a5f

    response=requests.get(url)

    
    return response.json()

@tool

def convert(base_currency_value: float, conversion_rate: Annotated[float,InjectedToolArg]) -> float:
    """ Given a base currency value and a conversion rate, returns the converted currency value."""
    return base_currency_value * conversion_rate

In [91]:
result=get_conversion_rate.invoke({"from_currency": "USD", "to_currency": "INR"})
print(result)

{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1770854401, 'time_last_update_utc': 'Thu, 12 Feb 2026 00:00:01 +0000', 'time_next_update_unix': 1770940801, 'time_next_update_utc': 'Fri, 13 Feb 2026 00:00:01 +0000', 'base_code': 'USD', 'target_code': 'INR', 'conversion_rate': 90.738}


In [92]:
Amount=convert.invoke({"base_currency_value": 100, "conversion_rate": result['conversion_rate']})

print(Amount)

9073.8


In [76]:
# Tool Biding with currency conversion tool

In [93]:
llm_with_tools=model.bind_tools([get_conversion_rate, convert])

In [78]:
messages=[HumanMessage(content="What is the conversion rate from USD to INR?, and based on that what is the value of 100 USD in INR?")]

In [80]:
ai_message=llm_with_tools.invoke(messages)
# print(ai_message.tool_calls)
messages.append(ai_message)


In [83]:
import json
for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get value of conversion rate
    if tool_call['name']=="get_conversion_rate":
        tool_result=get_conversion_rate.invoke(tool_call)
        conversion_rate =json.loads(tool_result.content)['conversion_rate']
        messages.append(tool_result)
    #    excecute the 2nd tool using conversionrate 
    if tool_call["name"]=="convert":

        tool_call["args"]["conversion_rate"]=conversion_rate
        tool_result=convert.invoke(tool_call["args"])
        print(tool_result)
        messages.append(tool_result)

    #     tool_result=convert.invoke(tool_call["args"])
    #     messages.append(tool_result)

In [94]:
messages = [HumanMessage(content="What is the conversion rate from USD to INR?, and based on that what is the value of 100 USD in INR?")]
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

# Variable to store state between tools
current_conversion_rate = 0.0

# 2. Process Tool Calls
if ai_message.tool_calls:
    for tool_call in ai_message.tool_calls:
        
        # --- Handle get_conversion_rate ---
        if tool_call['name'] == "get_conversion_rate":
            # Invoke the tool
            # Passing the full 'tool_call' dict lets LangChain automatically create a ToolMessage
            tool_msg = get_conversion_rate.invoke(tool_call)
            
            # Parse the result to save the rate for the next tool
            content_json = json.loads(tool_msg.content)
            current_conversion_rate = content_json['conversion_rate']
            
            messages.append(tool_msg)

        # --- Handle convert ---
        elif tool_call["name"] == "convert":
            # Inject the argument manually
            tool_args = tool_call["args"]
            tool_args["conversion_rate"] = current_conversion_rate
            
            # Execute the actual logic
            result_value = convert.invoke(tool_args)
            
            # CRITICAL STEP: Create the ToolMessage manually
            # We must link this result to the tool_call_id requested by the LLM
            tool_msg = ToolMessage(
                tool_call_id=tool_call['id'],
                content=str(result_value),
                name=tool_call['name']
            )
            
            messages.append(tool_msg)

In [95]:
print(llm_with_tools.invoke(messages))

content='The conversion rate from USD to INR is 90.738. The value of 100 USD in INR is 9073.8.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019c51e9-fa8e-7073-a5b0-f415ecf2a848-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 418, 'output_tokens': 35, 'total_tokens': 453, 'input_token_details': {'cache_read': 0}}
